# Lluvias máximas: ajuste de Gumbel (datos ficticios)
Este cuaderno hace TODO el cálculo. El documento `informe.pltx` no tiene celdas: lo vincula con `\pynotebook{}` y usa sus variables y funciones.

Se puede abrir y ejecutar en Jupyter o en VS Code como cualquier cuaderno.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Rutas relativas a la carpeta del cuaderno (Pyx lo ejecuta desde aquí).
serie = pd.read_csv('../datos/maximos_anuales.csv')
n_anos = len(serie)
media = serie['p_max_mm'].mean()
desv = serie['p_max_mm'].std()
serie.describe()

## Ajuste por el método de los momentos

In [ ]:
alfa = np.sqrt(6) * desv / np.pi          # parámetro de escala
u = media - 0.5772 * alfa                 # parámetro de posición

def cuantil(T):
    """Precipitación máxima diaria (mm) para un periodo de retorno T (años)."""
    return u - alfa * np.log(-np.log(1 - 1 / T))

periodos = [2, 5, 10, 25, 50, 100, 500]
cuantiles = pd.DataFrame({'T': periodos, 'P': [cuantil(T) for T in periodos]}).set_index('T')
cuantiles

In [ ]:
def filas_periodos(lista):
    """Filas de una tabla LaTeX: periodo, probabilidad anual y precipitación."""
    filas = []
    for T in lista:
        filas.append(f'{T} & {100 / T:.1f} & {cuantil(T):.1f} \\\\')
    return '\n'.join(filas)

def intensidad(T, duracion_min):
    """Intensidad media (mm/h) para una duración, con una curva IDF inventada."""
    Id = cuantil(T) / 24
    return Id * 11 ** ((28 ** 0.1 - (duracion_min / 60) ** 0.1) / (28 ** 0.1 - 1))

print(filas_periodos([2, 10, 100]))

## Gráfica
Se guarda como imagen en `figuras/`; el documento la incluye con `\includegraphics`.

In [ ]:
orden = np.sort(serie['p_max_mm'].values)
F_emp = np.arange(1, n_anos + 1) / (n_anos + 1)
T_emp = 1 / (1 - F_emp)
T_curva = np.logspace(np.log10(1.05), np.log10(500), 200)
fig, ax = plt.subplots(figsize=(6.4, 3.4))
ax.semilogx(T_curva, cuantil(T_curva), color='#007acc', lw=2, label='Gumbel (momentos)')
ax.semilogx(T_emp, orden, 'o', ms=4, color='#cf222e', label='serie anual')
ax.set_xlabel('periodo de retorno T (años)')
ax.set_ylabel('P máx. diaria (mm)')
ax.grid(alpha=0.3, which='both'); ax.legend(frameon=False)
fig.tight_layout()
import os
os.makedirs('figuras', exist_ok=True)
fig.savefig('figuras/gumbel.png', dpi=160)
tendencia = np.polyfit(serie['anio'], serie['p_max_mm'], 1)[0]   # mm/año